In [ ]:
import torch
import torch.nn as nn
import pytorch_lightning as pl
from typing import Optional, Tuple, Dict, Any
from torch.optim.lr_scheduler import ReduceLROnPlateau
import numpy as np
from scipy.special import gamma

class VaRNetBase(pl.LightningModule):
    """Base class for VaR estimation models"""

    def __init__(
        self,
        input_size: int,
        hidden_size: int,
        seq_length: int,
        num_layers: int = 2,
        dropout: float = 0.1,
        learning_rate: float = 1e-3,
        alpha: float = 0.025  # VaR significance level
    ):
        super().__init__()
        self.save_hyperparameters()

        # Core LSTM architecture
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=True
        )

        # Shared feature extraction
        self.feature_extractor = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.LayerNorm(hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, hidden_size // 4),
            nn.LayerNorm(hidden_size // 4),
            nn.GELU(),
            nn.Dropout(dropout)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Base forward pass"""
        lstm_out, _ = self.lstm(x)
        features = self.feature_extractor(lstm_out[:, -1])
        return features

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.parameters(),
            lr=self.hparams.learning_rate,
            weight_decay=0.01
        )
        scheduler = ReduceLROnPlateau(
            optimizer,
            mode='min',
            factor=0.5,
            patience=5,
            verbose=True
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "train_loss"
            }
        }

class SkewedStudentTVaRNet(VaRNetBase):
    """VaR estimation using Skewed Student-t Distribution"""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        hidden_size = self.hparams.hidden_size // 4

        # Volatility estimator (σ²)
        self.vol_estimator = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.LayerNorm(hidden_size // 2),
            nn.GELU(),
            nn.Linear(hidden_size // 2, 1),
            nn.Softplus()
        )

        # Skewness estimator (λ)
        self.skew_estimator = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.LayerNorm(hidden_size // 2),
            nn.GELU(),
            nn.Linear(hidden_size // 2, 1),
            nn.Tanh()  # Bounds skewness to [-1, 1]
        )

        # Degrees of freedom estimator (ν)
        self.df_estimator = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.LayerNorm(hidden_size // 2),
            nn.GELU(),
            nn.Linear(hidden_size // 2, 1),
            nn.Softplus()
        )

        # Exponential smoothing for volatility
        self.register_buffer('vol_ema', torch.tensor(0.0))
        self.ema_alpha = 0.97

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        features = super().forward(x)

        # Estimate parameters with constraints
        vol = self.vol_estimator(features)
        vol = torch.clamp(vol, min=1e-6, max=1.0)

        skew = self.skew_estimator(features)
        skew = torch.clamp(skew, min=-0.99, max=0.99)

        df = self.df_estimator(features)
        df = 2.1 + torch.clamp(df, min=0.0, max=297.9)  # Ensure df > 2

        # Update volatility EMA during training
        if self.training:
            with torch.no_grad():
                self.vol_ema = self.ema_alpha * self.vol_ema + (1 - self.ema_alpha) * vol.mean()

        return {
            'volatility': vol,
            'skewness': skew,
            'df': df
        }

    def skst_loglikelihood(self, returns: torch.Tensor, params: Dict[str, torch.Tensor]) -> torch.Tensor:
        """Compute negative log-likelihood for skewed Student-t distribution"""
        vol = params['volatility']
        skew = params['skewness']
        df = params['df']

        # Compute standardized returns
        z = returns / torch.sqrt(vol)

        # Constants for skewed t distribution
        c = torch.lgamma((df + 1)/2) - torch.lgamma(df/2) - 0.5 * torch.log(torch.pi * (df - 2))
        a = 4 * skew * torch.exp(c) * (df - 2) / (df - 1)
        b = torch.sqrt(1 + 3 * torch.square(skew) - torch.square(a))

        # Modified returns
        z_mod = (b * z + a) / (1 + torch.sign(z + a/b) * skew)

        # Log-likelihood
        ll = (torch.log(b) + c - 0.5 * torch.log(vol) -
              (df + 1)/2 * torch.log(1 + z_mod.square()/(df - 2)))

        return -ll.mean()

    def training_step(self, batch: Tuple[torch.Tensor, torch.Tensor], batch_idx: int) -> torch.Tensor:
        x, returns = batch
        params = self(x)

        # Main loss: negative log-likelihood
        loss = self.skst_loglikelihood(returns, params)

        # Regularization
        vol_reg = 0.1 * torch.mean((params['volatility'] - self.vol_ema).square())
        df_reg = 0.01 * torch.mean((params['df'] - 10).square())
        skew_reg = 0.01 * torch.mean(params['skewness'].square())

        total_loss = loss + vol_reg + df_reg + skew_reg

        # Logging
        self.log_dict({
            'train_loss': total_loss,
            'nll_loss': loss,
            'vol_reg': vol_reg,
            'df_reg': df_reg,
            'skew_reg': skew_reg,
            'mean_vol': params['volatility'].mean(),
            'mean_df': params['df'].mean(),
            'mean_skew': params['skewness'].mean()
        }, prog_bar=True)

        return total_loss

    def predict_var(self, x: torch.Tensor) -> Dict[str, float]:
        """Predict VaR and distribution parameters"""
        self.eval()
        with torch.no_grad():
            params = self(x)

            # Convert to numpy for scipy quantile function
            vol = params['volatility'].cpu().numpy()[0, 0]
            skew = params['skewness'].cpu().numpy()[0, 0]
            df = params['df'].cpu().numpy()[0, 0]

            # Calculate VaR using skewed t distribution quantile
            from arch.univariate import SkewStudent
            dist = SkewStudent()
            ppf = dist.ppf(self.hparams.alpha, parameters=[df, skew])
            var = np.sqrt(vol) * ppf

            return {
                'VaR': float(var),
                'volatility': float(vol),
                'skewness': float(skew),
                'df': float(df)
            }

ModuleNotFoundError: No module named 'pytorch_lightning'

In [ ]:

    def gather_prediction(self, prediction):
        self.df.loc[self.df.index[self.training_length + self.test_case], 'VaR'] = prediction
        print("Date: {0} -> RR: {1} | VaR: {2}".format(*(self.df.index[self.training_length + self.test_case],) + tuple(
            self.df.loc[self.df.index[self.training_length + self.test_case], ['log_returns', 'VaR']])))